
# Multi-maturity Arbitrage-Consistent Projection on bid--ask quotes

This Colab notebook extends the bid--ask ACP code to several maturities.

For maturities

$$
0<T_1<\cdots<T_m=\bar T,
$$

the state is a path vector \(s=(s_1,\ldots,s_m)\). A payoff received at \(T_j\) is accumulated to the common horizon \(\bar T\) by

$$
A_j=\exp(r(\bar T-T_j)).
$$

The corrected bid--ask system is required to contain an internal frictionless multi-maturity price system generated by a strictly positive dual certificate. The certificate variables are

$$
\lambda_v>0,\quad v\in V=X_1\times\cdots\times X_m,
\qquad
\beta_j>0,
$$

and they satisfy the multi-maturity analog of the single-maturity finite payoff equations.

The output is not one corrected mid price. The output is a corrected bid--ask system:

$$
C^{bid,ACP}_{j,i},\ C^{ask,ACP}_{j,i},\ P^{bid,ACP}_{j,i},\ P^{ask,ACP}_{j,i}.
$$

The objective minimizes a weighted \(L^1\) correction of all bid and ask endpoints.

As in the detection notebook, the Cartesian grid can grow quickly. For real data, keep only a small number of strikes per maturity unless only two maturities are used.


## Cell 1 — Install the required packages

This cell installs the numerical, plotting, and Yahoo Finance packages used by the notebook.

In [ ]:
# Install the external packages required by the notebook.
# This cell is intentionally small so that it is easy to re-run in Colab.
%pip -q install numpy pandas scipy matplotlib yfinance


## Cell 2 — Load the multi-maturity ACP implementation

This cell defines the synthetic data generator, finite path grid, multi-maturity ACP LP, diagnostics, tables, plots, and optional Yahoo Finance helpers.

In [ ]:

# ============================================================
# Multi-maturity Arbitrage-Consistent Projection on bid-ask quotes
# ============================================================
#
# This code extends the one-maturity ACP bid-ask projection to
# several maturities.
#
# The corrected output is a bid-ask system for all calls and puts
# at all maturities. The LP also constructs an internal strictly
# positive dual certificate:
#
#     lambda_v > 0 for every finite path-grid point v,
#     beta_j > 0 for every maturity T_j.
#
# The certificate proves deterministic static consistency of the
# corrected multi-maturity bid-ask system in the generated payoff
# space.
#
# ============================================================

from __future__ import annotations

import itertools
import math
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Dict, List, Optional, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog
from scipy.stats import norm


# ============================================================
# Data container
# ============================================================

@dataclass
class ACPMultiMaturityInput:
    """
    Container for a multi-maturity bid-ask ACP input.

    Each element of chains is a dictionary with two DataFrames:
        chains[j]["calls"] and chains[j]["puts"].

    Both DataFrames must contain columns:
        strike, bid, ask.
    """

    S0: float
    r: float
    maturities: List[float]
    chains: List[Dict[str, pd.DataFrame]]
    name: str = "multi-maturity ACP input"

    def __post_init__(self):
        self.S0 = float(self.S0)
        self.r = float(self.r)
        self.maturities = [float(t) for t in self.maturities]
        if self.S0 <= 0:
            raise ValueError("S0 must be positive.")
        if len(self.maturities) != len(self.chains):
            raise ValueError("maturities and chains must have the same length.")
        if any(t <= 0 for t in self.maturities):
            raise ValueError("All maturities must be positive.")
        if not np.all(np.diff(self.maturities) >= 0):
            raise ValueError("Maturities must be sorted in increasing order.")


# ============================================================
# Black-Scholes prices for synthetic tests
# ============================================================

def bs_call_put(S: float, K, r: float, sigma: float, tau: float):
    """
    Compute Black-Scholes European call and put prices.

    This function is used only for synthetic tests. The ACP LP
    itself is model-free and only enforces deterministic static
    consistency.
    """
    K = np.asarray(K, dtype=float)
    S = float(S)
    r = float(r)
    sigma = float(sigma)
    tau = float(tau)

    if tau <= 0:
        call = np.maximum(S - K, 0.0)
        put = np.maximum(K - S, 0.0)
        return call, put

    vol_sqrt = sigma * np.sqrt(tau)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * tau) / vol_sqrt
    d2 = d1 - vol_sqrt
    call = S * norm.cdf(d1) - K * np.exp(-r * tau) * norm.cdf(d2)
    put = K * np.exp(-r * tau) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return call, put


def make_disturbed_multi_maturity_bid_ask_test(
    S0: float = 100.0,
    r: float = 0.03,
    sigma: float = 0.20,
    maturities: Sequence[float] = (0.25, 0.50, 1.00),
    seed: int = 123,
) -> ACPMultiMaturityInput:
    """
    Generate synthetic multi-maturity bid-ask option quotes and
    deliberately perturb them.

    Replace this function with data imported from a CSV file,
    Excel file, database, Bloomberg, Yahoo Finance, or another
    pricing engine in real applications.
    """
    rng = np.random.default_rng(seed)
    chains = []
    K = np.arange(80.0, 121.0, 10.0)

    for j, tau in enumerate(maturities):
        call_true, put_true = bs_call_put(S0, K, r, sigma, tau)

        call_spread = 0.04 * np.maximum(call_true, 1.0) + 0.05
        put_spread = 0.04 * np.maximum(put_true, 1.0) + 0.05

        call_mid = call_true + rng.normal(0.0, 0.20, len(K))
        put_mid = put_true + rng.normal(0.0, 0.20, len(K))
        call_mid = np.maximum(call_mid, 0.01)
        put_mid = np.maximum(put_mid, 0.01)

        call_bid = np.maximum(call_mid - 0.5 * call_spread, 0.0)
        call_ask = call_mid + 0.5 * call_spread
        put_bid = np.maximum(put_mid - 0.5 * put_spread, 0.0)
        put_ask = put_mid + 0.5 * put_spread

        # Deliberate crossed-market and static-consistency distortions.
        if j == 1 and len(K) >= 5:
            call_bid[2] = call_ask[2] + 0.60
            put_bid[3] = put_ask[3] + 0.45
        if j == len(maturities) - 1 and len(K) >= 5:
            call_bid[1] += 1.00
            put_ask[4] = max(put_ask[4] - 0.90, 0.0)

        chains.append({
            "calls": pd.DataFrame({"strike": K, "bid": call_bid, "ask": call_ask}),
            "puts": pd.DataFrame({"strike": K, "bid": put_bid, "ask": put_ask}),
        })

    return ACPMultiMaturityInput(
        S0=S0,
        r=r,
        maturities=list(maturities),
        chains=chains,
        name="synthetic disturbed multi-maturity bid-ask input",
    )


# ============================================================
# Quote normalization and finite grid
# ============================================================

def _normalize_side(df: pd.DataFrame) -> pd.DataFrame:
    """Return a clean DataFrame with strike, bid, ask."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["strike", "bid", "ask"])
    out = df[["strike", "bid", "ask"]].copy()
    out["strike"] = out["strike"].astype(float)
    out["bid"] = np.maximum(out["bid"].astype(float), 0.0)
    out["ask"] = np.maximum(out["ask"].astype(float), 0.0)
    out = out.sort_values("strike").reset_index(drop=True)
    return out


def _union_strikes(chain: Dict[str, pd.DataFrame]) -> np.ndarray:
    """Return sorted union of call and put strikes for one maturity."""
    values = []
    for side in ["calls", "puts"]:
        df = _normalize_side(chain.get(side))
        values.extend(df["strike"].tolist())
    if not values:
        return np.array([], dtype=float)
    return np.array(sorted(set(np.round(values, 10))), dtype=float)


def build_multi_maturity_grid(data: ACPMultiMaturityInput, max_grid_points: int = 20000):
    """
    Build V = X_1 x ... x X_m, where X_j contains zero and all
    strikes traded at maturity T_j.
    """
    node_sets = []
    for chain in data.chains:
        K = _union_strikes(chain)
        node_sets.append(np.concatenate(([0.0], K)))

    grid_size = int(np.prod([len(x) for x in node_sets]))
    if grid_size > max_grid_points:
        raise ValueError(
            f"The Cartesian grid has {grid_size} points, which exceeds max_grid_points={max_grid_points}. "
            "Reduce the number of maturities or strikes per maturity."
        )

    vertices = np.array(list(itertools.product(*node_sets)), dtype=float)
    return node_sets, vertices


# ============================================================
# Main ACP LP
# ============================================================

def acp_multi_maturity_bid_ask(
    data: ACPMultiMaturityInput,
    weights: Optional[Dict[str, List[np.ndarray]]] = None,
    lambda_floor: float = 1e-12,
    beta_floor: float = 1e-12,
    max_grid_points: int = 20000,
    return_certificate: bool = True,
) -> Dict:
    """
    Run multi-maturity ACP directly on bid-ask quotes.

    The corrected bid-ask system is required to contain an internal
    arbitrage-free frictionless multi-maturity price system. This is
    enforced through a strictly positive dual certificate.

    Certificate equations
    ---------------------
    Let V be the Cartesian product of all maturity-specific strike
    grids. The variables are lambda_v and beta_j. They satisfy

        sum_v lambda_v = exp(-r T_bar),

    and, for each maturity j,

        A_j * sum_v lambda_v v_j + beta_j = S0.

    Internal representative values are

        C_tilde_{j,i} = A_j * sum_v lambda_v (v_j - K_{j,i})^+ + beta_j,
        P_tilde_{j,i} = A_j * sum_v lambda_v (K_{j,i} - v_j)^+.

    The corrected bid-ask intervals must contain these values.
    """
    m = len(data.maturities)
    T_bar = max(data.maturities)
    A = np.exp(data.r * (T_bar - np.asarray(data.maturities, dtype=float)))
    df_bar = math.exp(-data.r * T_bar)

    calls = [_normalize_side(chain.get("calls")) for chain in data.chains]
    puts = [_normalize_side(chain.get("puts")) for chain in data.chains]
    node_sets, vertices = build_multi_maturity_grid(data, max_grid_points=max_grid_points)
    L = len(vertices)

    if L * lambda_floor >= df_bar:
        raise ValueError(
            "lambda_floor is too large for the number of grid points. "
            "Use a smaller lambda_floor or reduce the grid size."
        )

    # If no weights are supplied, use equal weights for all endpoints.
    if weights is None:
        weights = {
            "call_bid": [np.ones(len(calls[j])) for j in range(m)],
            "call_ask": [np.ones(len(calls[j])) for j in range(m)],
            "put_bid": [np.ones(len(puts[j])) for j in range(m)],
            "put_ask": [np.ones(len(puts[j])) for j in range(m)],
        }

    def _weight_list(key, n_list):
        if key not in weights:
            return [np.ones(n) for n in n_list]
        out = []
        for arr, n in zip(weights[key], n_list):
            arr = np.asarray(arr, dtype=float)
            if arr.shape != (n,):
                raise ValueError(f"weights['{key}'] has an array with wrong length.")
            if np.any(arr < 0):
                raise ValueError(f"weights['{key}'] must be nonnegative.")
            out.append(arr)
        return out

    n_calls = [len(df) for df in calls]
    n_puts = [len(df) for df in puts]
    w_cb = _weight_list("call_bid", n_calls)
    w_ca = _weight_list("call_ask", n_calls)
    w_pb = _weight_list("put_bid", n_puts)
    w_pa = _weight_list("put_ask", n_puts)

    # --------------------------------------------------------
    # Variable indexing
    # --------------------------------------------------------
    # z contains:
    #   lambda_v for all vertices v in V
    #   beta_j for all maturities
    #   corrected call bid and ask endpoints
    #   corrected put bid and ask endpoints
    #   absolute deviations for every corrected endpoint
    # --------------------------------------------------------

    idx = {}
    start = 0

    idx["lambda"] = slice(start, start + L)
    start += L
    idx["beta"] = slice(start, start + m)
    start += m

    for key, n_list in [("cb", n_calls), ("ca", n_calls), ("pb", n_puts), ("pa", n_puts),
                        ("dcb", n_calls), ("dca", n_calls), ("dpb", n_puts), ("dpa", n_puts)]:
        idx[key] = []
        for n in n_list:
            idx[key].append(slice(start, start + n))
            start += n

    num_vars = start

    # --------------------------------------------------------
    # Objective: weighted L1 correction of all endpoints.
    # --------------------------------------------------------
    c = np.zeros(num_vars)
    for j in range(m):
        c[idx["dcb"][j]] = w_cb[j]
        c[idx["dca"][j]] = w_ca[j]
        c[idx["dpb"][j]] = w_pb[j]
        c[idx["dpa"][j]] = w_pa[j]

    # --------------------------------------------------------
    # Equality constraints for the internal certificate.
    # --------------------------------------------------------
    A_eq = []
    b_eq = []

    # sum_v lambda_v = exp(-r T_bar)
    row = np.zeros(num_vars)
    row[idx["lambda"]] = 1.0
    A_eq.append(row)
    b_eq.append(df_bar)

    # For each maturity j: A_j * sum_v lambda_v v_j + beta_j = S0.
    for j in range(m):
        row = np.zeros(num_vars)
        row[idx["lambda"]] = A[j] * vertices[:, j]
        row[idx["beta"].start + j] = 1.0
        A_eq.append(row)
        b_eq.append(data.S0)

    # --------------------------------------------------------
    # Inequality constraints.
    # --------------------------------------------------------
    A_ub = []
    b_ub = []

    def add_abs_constraints(price_slice, dev_slice, raw):
        """Add d >= |corrected - raw| in linear form."""
        raw = np.asarray(raw, dtype=float)
        for i in range(len(raw)):
            row = np.zeros(num_vars)
            row[price_slice.start + i] = 1.0
            row[dev_slice.start + i] = -1.0
            A_ub.append(row)
            b_ub.append(raw[i])

            row = np.zeros(num_vars)
            row[price_slice.start + i] = -1.0
            row[dev_slice.start + i] = -1.0
            A_ub.append(row)
            b_ub.append(-raw[i])

    for j in range(m):
        # Call bid <= call ask.
        for i in range(n_calls[j]):
            row = np.zeros(num_vars)
            row[idx["cb"][j].start + i] = 1.0
            row[idx["ca"][j].start + i] = -1.0
            A_ub.append(row)
            b_ub.append(0.0)

        # Put bid <= put ask.
        for i in range(n_puts[j]):
            row = np.zeros(num_vars)
            row[idx["pb"][j].start + i] = 1.0
            row[idx["pa"][j].start + i] = -1.0
            A_ub.append(row)
            b_ub.append(0.0)

        # Internal call certificates inside corrected intervals.
        for i, K in enumerate(calls[j]["strike"].to_numpy(dtype=float)):
            payoff = A[j] * np.maximum(vertices[:, j] - K, 0.0)

            # C_bid_ACP <= C_tilde.
            row = np.zeros(num_vars)
            row[idx["cb"][j].start + i] = 1.0
            row[idx["lambda"]] = -payoff
            row[idx["beta"].start + j] = -1.0
            A_ub.append(row)
            b_ub.append(0.0)

            # C_tilde <= C_ask_ACP.
            row = np.zeros(num_vars)
            row[idx["lambda"]] = payoff
            row[idx["beta"].start + j] = 1.0
            row[idx["ca"][j].start + i] = -1.0
            A_ub.append(row)
            b_ub.append(0.0)

        # Internal put certificates inside corrected intervals.
        for i, K in enumerate(puts[j]["strike"].to_numpy(dtype=float)):
            payoff = A[j] * np.maximum(K - vertices[:, j], 0.0)

            # P_bid_ACP <= P_tilde.
            row = np.zeros(num_vars)
            row[idx["pb"][j].start + i] = 1.0
            row[idx["lambda"]] = -payoff
            A_ub.append(row)
            b_ub.append(0.0)

            # P_tilde <= P_ask_ACP.
            row = np.zeros(num_vars)
            row[idx["lambda"]] = payoff
            row[idx["pa"][j].start + i] = -1.0
            A_ub.append(row)
            b_ub.append(0.0)

        # Absolute deviations for all corrected endpoints.
        add_abs_constraints(idx["cb"][j], idx["dcb"][j], calls[j]["bid"].to_numpy(dtype=float))
        add_abs_constraints(idx["ca"][j], idx["dca"][j], calls[j]["ask"].to_numpy(dtype=float))
        add_abs_constraints(idx["pb"][j], idx["dpb"][j], puts[j]["bid"].to_numpy(dtype=float))
        add_abs_constraints(idx["pa"][j], idx["dpa"][j], puts[j]["ask"].to_numpy(dtype=float))

    # --------------------------------------------------------
    # Variable bounds.
    # --------------------------------------------------------
    bounds = []

    # Positive lower bounds for lambda variables.
    for _ in range(L):
        bounds.append((lambda_floor, None))

    # Positive lower bounds for beta variables.
    for _ in range(m):
        bounds.append((beta_floor, None))

    # Corrected quotes and absolute deviations are nonnegative.
    remaining = num_vars - L - m
    for _ in range(remaining):
        bounds.append((0.0, None))

    # --------------------------------------------------------
    # Solve the LP.
    # --------------------------------------------------------
    res = linprog(
        c,
        A_ub=np.asarray(A_ub),
        b_ub=np.asarray(b_ub),
        A_eq=np.asarray(A_eq),
        b_eq=np.asarray(b_eq),
        bounds=bounds,
        method="highs",
    )

    if not res.success:
        raise RuntimeError(f"Multi-maturity ACP LP failed: {res.message}")

    z = res.x

    result = {
        "success": res.success,
        "message": res.message,
        "objective": float(res.fun),
        "data": data,
        "idx": idx,
        "z": z,
        "vertices": vertices,
        "node_sets": node_sets,
        "A": A,
        "T_bar": T_bar,
        "calls_raw": calls,
        "puts_raw": puts,
        "call_bid_acp": [z[idx["cb"][j]] for j in range(m)],
        "call_ask_acp": [z[idx["ca"][j]] for j in range(m)],
        "put_bid_acp": [z[idx["pb"][j]] for j in range(m)],
        "put_ask_acp": [z[idx["pa"][j]] for j in range(m)],
    }

    if return_certificate:
        lam = z[idx["lambda"]]
        beta = z[idx["beta"]]
        result["lambda"] = lam
        result["beta"] = beta

        certificate_calls = []
        certificate_puts = []
        for j in range(m):
            cc = []
            for K in calls[j]["strike"].to_numpy(dtype=float):
                cc.append(A[j] * np.dot(lam, np.maximum(vertices[:, j] - K, 0.0)) + beta[j])
            pc = []
            for K in puts[j]["strike"].to_numpy(dtype=float):
                pc.append(A[j] * np.dot(lam, np.maximum(K - vertices[:, j], 0.0)))
            certificate_calls.append(np.asarray(cc))
            certificate_puts.append(np.asarray(pc))
        result["certificate_call"] = certificate_calls
        result["certificate_put"] = certificate_puts

    return result


# ============================================================
# Diagnostics and tables
# ============================================================

def build_multi_maturity_acp_table(result: Dict) -> pd.DataFrame:
    """
    Build one table containing raw and ACP-corrected quotes.
    """
    data = result["data"]
    calls = result["calls_raw"]
    puts = result["puts_raw"]
    rows = []

    for j, tau in enumerate(data.maturities):
        for i, row in calls[j].iterrows():
            rows.append({
                "maturity_index": j + 1,
                "T": tau,
                "type": "call",
                "strike": row["strike"],
                "raw_bid": row["bid"],
                "raw_ask": row["ask"],
                "acp_bid": result["call_bid_acp"][j][i],
                "acp_ask": result["call_ask_acp"][j][i],
                "certificate": result.get("certificate_call", [None] * len(data.maturities))[j][i]
                    if "certificate_call" in result else np.nan,
            })
        for i, row in puts[j].iterrows():
            rows.append({
                "maturity_index": j + 1,
                "T": tau,
                "type": "put",
                "strike": row["strike"],
                "raw_bid": row["bid"],
                "raw_ask": row["ask"],
                "acp_bid": result["put_bid_acp"][j][i],
                "acp_ask": result["put_ask_acp"][j][i],
                "certificate": result.get("certificate_put", [None] * len(data.maturities))[j][i]
                    if "certificate_put" in result else np.nan,
            })
    return pd.DataFrame(rows)


def check_multi_maturity_acp_result(result: Dict, tol: float = 1e-8):
    """
    Print diagnostic checks for a multi-maturity ACP result.
    """
    print("ACP status:", result["message"])
    print("ACP objective:", result["objective"])
    print("Number of path-grid vertices:", len(result["vertices"]))
    print()

    all_ok = True
    for j in range(len(result["data"].maturities)):
        cb = result["call_bid_acp"][j]
        ca = result["call_ask_acp"][j]
        pb = result["put_bid_acp"][j]
        pa = result["put_ask_acp"][j]

        ok_call_spread = np.all(cb <= ca + tol)
        ok_put_spread = np.all(pb <= pa + tol)
        ok_nonneg = np.all(cb >= -tol) and np.all(ca >= -tol) and np.all(pb >= -tol) and np.all(pa >= -tol)

        print(f"Maturity {j + 1}, T={result['data'].maturities[j]:.4f}")
        print("  call bid <= call ask:", ok_call_spread)
        print("  put bid <= put ask:", ok_put_spread)
        print("  corrected quotes nonnegative:", ok_nonneg)

        all_ok = all_ok and ok_call_spread and ok_put_spread and ok_nonneg

        if "certificate_call" in result:
            cc = result["certificate_call"][j]
            pc = result["certificate_put"][j]
            ok_cc = np.all((cb <= cc + tol) & (cc <= ca + tol)) if len(cc) else True
            ok_pc = np.all((pb <= pc + tol) & (pc <= pa + tol)) if len(pc) else True
            print("  call certificate inside corrected intervals:", ok_cc)
            print("  put certificate inside corrected intervals:", ok_pc)
            all_ok = all_ok and ok_cc and ok_pc
        print()

    if "lambda" in result:
        print("Minimum lambda:", float(np.min(result["lambda"])))
        print("Minimum beta:", float(np.min(result["beta"])))

    print("Overall diagnostic conclusion:", "passed" if all_ok else "failed")


def plot_acp_corrections_by_maturity(result: Dict):
    """
    Plot raw and ACP-corrected mid prices by maturity and option type.
    """
    table = build_multi_maturity_acp_table(result)
    table["raw_mid"] = 0.5 * (table["raw_bid"] + table["raw_ask"])
    table["acp_mid"] = 0.5 * (table["acp_bid"] + table["acp_ask"])

    for (j, opt_type), sub in table.groupby(["maturity_index", "type"]):
        plt.figure(figsize=(8, 4))
        plt.plot(sub["strike"], sub["raw_mid"], marker="o", label="raw mid")
        plt.plot(sub["strike"], sub["acp_mid"], marker="x", label="ACP mid")
        plt.title(f"{opt_type.capitalize()} corrections, maturity {j}")
        plt.xlabel("Strike")
        plt.ylabel("Mid price")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()


# ============================================================
# Optional weights based on raw spreads
# ============================================================

def inverse_spread_weights_multi(data: ACPMultiMaturityInput, floor: float = 1e-4) -> Dict[str, List[np.ndarray]]:
    """
    Create endpoint weights inversely proportional to bid-ask spreads.

    Wider raw spreads receive smaller weights, so the ACP is allowed
    to move less reliable quotes more easily.
    """
    weights = {"call_bid": [], "call_ask": [], "put_bid": [], "put_ask": []}
    for chain in data.chains:
        calls = _normalize_side(chain.get("calls"))
        puts = _normalize_side(chain.get("puts"))
        call_spread = np.maximum(calls["ask"].to_numpy(dtype=float) - calls["bid"].to_numpy(dtype=float), floor)
        put_spread = np.maximum(puts["ask"].to_numpy(dtype=float) - puts["bid"].to_numpy(dtype=float), floor)
        weights["call_bid"].append(1.0 / call_spread)
        weights["call_ask"].append(1.0 / call_spread)
        weights["put_bid"].append(1.0 / put_spread)
        weights["put_ask"].append(1.0 / put_spread)
    return weights


# ============================================================
# Yahoo helper for real data
# ============================================================

def _clean_yahoo_side(df: pd.DataFrame, max_rows: Optional[int], S0: float) -> pd.DataFrame:
    """Clean one Yahoo option side for ACP input."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["strike", "bid", "ask"])
    out = df.copy()
    for col in ["strike", "bid", "ask", "lastPrice"]:
        if col not in out.columns:
            out[col] = np.nan
    out = out[["strike", "bid", "ask", "lastPrice"]].copy()
    out = out.dropna(subset=["strike"])
    out["strike"] = out["strike"].astype(float)
    out["bid"] = pd.to_numeric(out["bid"], errors="coerce").fillna(0.0)
    out["ask"] = pd.to_numeric(out["ask"], errors="coerce")
    out["lastPrice"] = pd.to_numeric(out["lastPrice"], errors="coerce")
    bad = out["ask"].isna() | (out["ask"] <= 0)
    out.loc[bad, "ask"] = out.loc[bad, "lastPrice"]
    out["ask"] = out["ask"].fillna(out["bid"])
    out["ask"] = np.maximum(out["ask"].to_numpy(dtype=float), out["bid"].to_numpy(dtype=float))
    out = out[(out["strike"] > 0) & np.isfinite(out["ask"])]
    out = out.sort_values("strike").reset_index(drop=True)
    if max_rows is not None and len(out) > max_rows:
        out["moneyness_distance"] = np.abs(out["strike"] / S0 - 1.0)
        out = out.sort_values("moneyness_distance").head(max_rows)
        out = out.sort_values("strike").drop(columns=["moneyness_distance"]).reset_index(drop=True)
    return out[["strike", "bid", "ask"]]


def fetch_yahoo_acp_multi_maturity_input(
    ticker: str,
    expiration_dates: Sequence[str],
    r: float = 0.03,
    max_options_per_side: int = 6,
) -> ACPMultiMaturityInput:
    """Download a small multi-maturity ACP input from Yahoo Finance."""
    import yfinance as yf

    tk = yf.Ticker(ticker)
    hist = tk.history(period="5d")
    if hist.empty:
        raise RuntimeError("Could not download a recent stock price from Yahoo Finance.")
    S0 = float(hist["Close"].dropna().iloc[-1])

    today = datetime.now(timezone.utc).date()
    maturities = []
    chains = []
    for exp in expiration_dates:
        exp_date = datetime.strptime(exp, "%Y-%m-%d").date()
        tau = max((exp_date - today).days / 365.25, 1.0 / 365.25)
        opt = tk.option_chain(exp)
        calls = _clean_yahoo_side(opt.calls, max_options_per_side, S0)
        puts = _clean_yahoo_side(opt.puts, max_options_per_side, S0)
        if len(calls) == 0 and len(puts) == 0:
            continue
        maturities.append(tau)
        chains.append({"calls": calls, "puts": puts})

    order = np.argsort(maturities)
    maturities = [maturities[i] for i in order]
    chains = [chains[i] for i in order]

    return ACPMultiMaturityInput(
        S0=S0,
        r=r,
        maturities=maturities,
        chains=chains,
        name=f"Yahoo ACP input for {ticker.upper()}",
    )


def show_yahoo_expirations(ticker: str):
    """Print the expiration dates currently available from Yahoo Finance."""
    import yfinance as yf
    tk = yf.Ticker(ticker)
    expirations = list(tk.options)
    if not expirations:
        raise RuntimeError("Yahoo Finance did not return expiration dates for this ticker.")
    print(f"Available expirations for {ticker.upper()}:")
    for k, exp in enumerate(expirations, start=1):
        print(f"{k:2d}. {exp}")
    return expirations


# ============================================================
# Demo and interactive workflows
# ============================================================

def run_synthetic_acp_demo():
    """Run the multi-maturity ACP workflow on synthetic data."""
    data = make_disturbed_multi_maturity_bid_ask_test()
    weights = inverse_spread_weights_multi(data)
    result = acp_multi_maturity_bid_ask(
        data,
        weights=weights,
        lambda_floor=1e-12,
        beta_floor=1e-12,
        max_grid_points=20000,
        return_certificate=True,
    )
    check_multi_maturity_acp_result(result)
    table = build_multi_maturity_acp_table(result)
    display(table)
    plot_acp_corrections_by_maturity(result)
    return result


def run_yahoo_acp_interactive():
    """
    Interactive Yahoo Finance workflow for multi-maturity ACP.
    """
    ticker = input("Ticker symbol, for example AAPL: ").strip().upper()
    expirations = show_yahoo_expirations(ticker)

    raw = input("Enter expiration numbers separated by commas, for example 1,2,3: ").strip()
    selected = []
    for token in raw.split(","):
        k = int(token.strip())
        selected.append(expirations[k - 1])

    r = float(input("Continuously compounded risk-free rate, for example 0.03: ").strip() or "0.03")
    max_side = int(input("Max options per side and maturity, for example 5 or 6: ").strip() or "6")

    data = fetch_yahoo_acp_multi_maturity_input(
        ticker=ticker,
        expiration_dates=selected,
        r=r,
        max_options_per_side=max_side,
    )
    weights = inverse_spread_weights_multi(data)
    result = acp_multi_maturity_bid_ask(
        data,
        weights=weights,
        lambda_floor=1e-12,
        beta_floor=1e-12,
        max_grid_points=20000,
        return_certificate=True,
    )
    check_multi_maturity_acp_result(result)
    table = build_multi_maturity_acp_table(result)
    display(table)
    plot_acp_corrections_by_maturity(result)
    return result


## Cell 3 — Run the synthetic ACP example

The synthetic example deliberately includes disturbed bid-ask quotes. For live Yahoo Finance data, call `run_yahoo_acp_interactive()` instead.

In [ ]:
# Run the synthetic multi-maturity ACP demo.
# For live Yahoo data, run: acp_results = run_yahoo_acp_interactive()
acp_results = run_synthetic_acp_demo()
